<div style="border-left:6px solid #ae0000; padding:6px 20px; margin-bottom:4px;">
<h1 style="margin:0; color:#0d2741;">Medidas de Tendencia Central</h1>
<p style="margin:2px 0 0 0; color:#0d2741; font-size:1.15em;">Media, Mediana y Moda</p>
<p style="margin:6px 0 0 0; color:#444; font-size:1.05em;"><em>Estadística Computacional para la Toma de Decisiones</em></p>
</div>

**Magíster en Ciencia de Datos e Inteligencia Artificial** · Universidad Andrés Bello  
Maidana, J.P. (2026)

---

> **Cómo usar este notebook.** Ejecuta las celdas en orden (de arriba hacia abajo). Comienza por la celda **«Preparación del entorno»**, que carga las librerías y fija el estilo de los gráficos. Las celdas de código reproducen el material del apunte y comparten un mismo kernel, por lo que conviene ejecutarlas en secuencia.

### Preparación del entorno

In [ ]:
# Librerías base usadas en todo el notebook
import numpy as np
import pandas as pd
from scipy import stats          # scipy.stats: usaremos stats.mode()
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# Estilo de gráficos coherente con el apunte
plt.rcParams.update({
    "figure.figsize": (10, 4.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "font.size": 11,
})

UNAB_NAVY = "#0d2741"
UNAB_RED  = "#ae0000"

# Compatibilidad de boxplot: 'orientation' existe desde matplotlib 3.10;
# en versiones anteriores se usa 'vert'. Esto evita errores según el entorno.
_MPL_GE_310 = tuple(int(x) for x in matplotlib.__version__.split(".")[:2]) >= (3, 10)
HBOX_KW = {"orientation": "horizontal"} if _MPL_GE_310 else {"vert": False}

print("Entorno listo")
print("numpy", np.__version__, "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__)

## 1. Introducción

Las medidas de tendencia central —media, mediana y moda— son herramientas fundamentales para resumir datos en un único número representativo. Sin embargo, su utilidad depende críticamente de **elegir la medida adecuada al contexto**: cada una cuenta una historia diferente, y seleccionar la incorrecta puede conducir a interpretaciones y decisiones erróneas.

Considérese el siguiente escenario: en una empresa, 9 empleados perciben salarios de \$45.000 y un ejecutivo recibe \$800.000. El salario promedio resultante sería \$116.500, cifra que **no representa a ninguno** de los trabajadores. En este caso, la media es una medida engañosa. La mediana, en cambio, sería \$45.000, reflejando el salario real del empleado típico.

Otro ejemplo: un gerente de operaciones debe reportar los tiempos de entrega. La mayoría de envíos llega en 2–3 días, pero un caso excepcional tardó 45 días. Si se reporta la **media**, el resultado sería 7 días, generando la percepción de un servicio lento. Si se reporta la **mediana**, el resultado sería 3 días, que corresponde a la experiencia real del cliente típico. Los datos son idénticos; la interpretación, radicalmente diferente.

Este apunte no se centra únicamente en las fórmulas —que son relativamente simples— sino en desarrollar el **criterio** para seleccionar la medida apropiada, interpretar correctamente sus resultados y comunicar hallazgos con rigor estadístico.

<div style="background-color:#fffceb; border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#f57f17; font-size:1.05em;">📝&nbsp; Nota — Objetivos de aprendizaje</p>

<ol style="margin:4px 0 0 0;">
<li>Dominar las definiciones matemáticas de media, mediana y moda.</li>
<li>Identificar cuándo utilizar cada medida según las características de los datos.</li>
<li>Implementarlas eficientemente con NumPy y pandas.</li>
<li>Interpretar la relación entre estas medidas para detectar asimetría y outliers.</li>
<li>Aplicar criterios de selección en escenarios reales de análisis de datos.</li>
<li>Comunicar resultados estadísticos con precisión y transparencia.</li>
</ol>
</div>

## 2. Panorama de las medidas de tendencia central

### 2.1 Mapa conceptual

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.set_xlim(0, 12); ax.set_ylim(1.4, 7.8); ax.axis("off")

def _box(x, y, w, h, text, fc, tc="white", fs=10):
    ax.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h,
                 boxstyle="round,pad=0.02,rounding_size=0.12",
                 fc=fc, ec="white", lw=1.5, zorder=2))
    ax.text(x, y, text, ha="center", va="center", color=tc,
            fontsize=fs, fontweight="bold", zorder=3)

def _line(x1, y1, x2, y2):
    ax.plot([x1, x2], [y1, y2], color="#888", lw=1.4, zorder=1)

_box(6, 7, 3.6, 0.9, "Medidas de\nTendencia Central", "#7b4fa3")
ramas = {"Media\n" + r"$\bar{x},\ \mu$":  (2.3, 4.7, "#2e6fb0"),
         "Mediana\n" + r"$Me,\ Q_2$":       (6.0, 4.7, "#2e8b57"),
         "Moda\n" + r"$Mo$":                 (9.7, 4.7, "#d98a00")}
for t, (x, y, c) in ramas.items():
    _line(6, 6.55, x, 5.15); _box(x, y, 2.6, 0.95, t, c)

hojas = [(1.3, "Media\naritmética", "#cfe0f0", "#13314f", 2.3),
         (3.3, "Media\nponderada", "#cfe0f0", "#13314f", 2.3),
         (5.0, "Percentil 50", "#cbe6d4", "#14502c", 6.0),
         (7.0, "Robusta a\noutliers", "#cbe6d4", "#14502c", 6.0),
         (8.7, "Uni/\nMultimodal", "#f6e2c0", "#7a4d00", 9.7),
         (10.7, "Para\ncategóricas", "#f6e2c0", "#7a4d00", 9.7)]
for x, t, fc, tc, xp in hojas:
    _line(xp, 4.22, x, 2.85); _box(x, 2.4, 1.85, 0.95, t, fc, tc, 8)

ax.set_title("Mapa conceptual de las medidas de tendencia central",
             fontsize=13, color=UNAB_NAVY, pad=8)
plt.tight_layout(); plt.show()

### 2.2 La media

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Media aritmética</p>

La media es la suma de todos los valores dividida por la cantidad de observaciones.

**Para una muestra:**

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i = \frac{x_1 + x_2 + \cdots + x_n}{n}$$

**Para una población completa:**

$$\mu = \frac{1}{N}\sum_{i=1}^{N} x_i$$

Notación: $\bar{x}$ denota la media muestral; $\mu$ (mu) la media poblacional; $n$ el tamaño de la muestra; $N$ el de la población.
</div>

#### Propiedades de la media

1. **Sensibilidad a todos los valores.** Cada observación contribuye por igual al cálculo. Un único valor extremo puede modificar sustancialmente la media, lo que constituye su principal limitación.
2. **Propiedad de balance.** La suma de las desviaciones respecto a la media es siempre cero: $\sum_{i=1}^{n}(x_i - \bar{x}) = 0$. La media actúa como el *centro de gravedad* de la distribución.
3. **Invarianza ante transformaciones lineales.** Si $y = a + bx$, entonces $\bar{y} = a + b\bar{x}$. Esto facilita convertir entre unidades de medida.
4. **Óptimo de mínimos cuadrados.** La media minimiza la suma de cuadrados de las desviaciones: $\bar{x} = \arg\min_{c}\sum_{i=1}^{n}(x_i - c)^2$. Es el fundamento de la regresión lineal.
5. **Unicidad.** Existe una única media para cada conjunto de datos, aunque ese valor no necesariamente pertenece al conjunto original.

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Contextos apropiados para la media</p>

La media es adecuada cuando los datos son aproximadamente simétricos y no contienen valores extremos dominantes:
<ul>
<li>Ticket de compra promedio en retail (sin pedidos corporativos excepcionales)</li>
<li>Retorno promedio de inversión financiera</li>
<li>CTR (<i>click-through rate</i>) promedio en campañas digitales</li>
<li>Tiempo de producción en procesos industriales controlados</li>
<li>Calificaciones académicas en distribuciones equilibradas</li>
<li>Temperatura promedio diaria</li>
</ul>
<b>Criterio general:</b> la media es apropiada cuando la distribución es razonablemente simétrica y los valores extremos no distorsionan la representatividad del resultado.
</div>

### 2.3 La mediana

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Mediana</p>

La mediana es el valor que ocupa la posición central cuando los datos están ordenados de menor a mayor. El 50 % de las observaciones se ubica por debajo y el otro 50 % por encima.

Para datos ordenados $x_{(1)} \leq x_{(2)} \leq \cdots \leq x_{(n)}$:

$$Me = \begin{cases} x_{\left(\frac{n+1}{2}\right)} & \text{si } n \text{ es impar} \\[0.3cm] \dfrac{x_{(n/2)} + x_{(n/2 + 1)}}{2} & \text{si } n \text{ es par} \end{cases}$$

Ejemplo: para $\{2, 3, 5, 7, 100\}$, la mediana es 5. El valor extremo 100 no la afecta, lo que ilustra su **robustez**.
</div>

#### Propiedades de la mediana

1. **Robustez ante valores extremos.** No se ve afectada por la magnitud de los valores en los extremos, solo por su posición. Esta es su principal ventaja respecto a la media.
2. **Equivalencia con el percentil 50.** Corresponde al segundo cuartil ($Q_2$). Si la mediana de edad es 35 años, el 50 % de los individuos tiene 35 años o menos.
3. **Óptimo de desviaciones absolutas.** Mientras la media minimiza las desviaciones cuadráticas, la mediana minimiza las absolutas: $Me = \arg\min_{c}\sum_{i=1}^{n}|x_i - c|$. Las desviaciones absolutas son menos sensibles a extremos, lo que explica su mayor robustez.
4. **Requiere ordenamiento.** Su cálculo exige ordenar los datos, operación de complejidad $O(n \log n)$, frente a $O(n)$ de la media. En conjuntos muy grandes esto puede ser relevante.

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Contextos apropiados para la mediana</p>

La mediana es la medida más adecuada cuando se busca el "valor típico" en presencia de asimetría u outliers:
<ul>
<li><b>Salarios:</b> los ingresos de ejecutivos distorsionan la media; la mediana representa mejor al trabajador típico.</li>
<li><b>Precios de viviendas:</b> las propiedades de lujo no deben dominar el indicador de precio representativo.</li>
<li><b>Tiempos de respuesta:</b> en sistemas web y APIs, los picos ocasionales no deben inflar el indicador típico.</li>
<li><b>Ventas diarias:</b> días excepcionales (ofertas masivas) no deben distorsionar el indicador habitual.</li>
<li><b>Calificaciones de productos:</b> las opiniones extremas no deberían dominar la valoración representativa.</li>
</ul>
<b>Criterio general:</b> cuando se requiere el "valor típico" en distribuciones asimétricas o con outliers, la mediana es la medida apropiada.
</div>

### 2.4 La moda

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Moda</p>

La moda es el valor que aparece con mayor frecuencia en un conjunto de datos. Es la **única** medida de tendencia central aplicable a variables nominales (categóricas sin orden).

**Clasificación según el número de modas:**
- **Unimodal:** una sola moda (un pico en el histograma).
- **Bimodal:** dos modas, posible indicador de dos subpoblaciones mezcladas.
- **Multimodal:** más de dos modas.
- **Amodal:** sin moda clara (todos los valores con frecuencia similar).

A diferencia de la media y la mediana, la moda puede no existir o no ser única.
</div>

#### Propiedades de la moda

1. **Aplicabilidad universal.** Puede calcularse con cualquier tipo de variable, incluidas las nominales (colores, categorías). No requiere que los valores sean sumables ni ordenables.
2. **Posible multiplicidad.** Cuando dos o más valores tienen la misma frecuencia máxima, todos son modas. Una distribución bimodal puede revelar dos grupos distintos mezclados.
3. **No requiere fórmula algebraica.** El cálculo es exclusivamente de conteo de frecuencias.
4. **Describe el comportamiento predominante.** Informa sobre lo que ocurre con mayor frecuencia: clave en gestión de inventarios, diseño de productos y análisis de preferencias.
5. **Insensibilidad a magnitudes.** Solo considera la frecuencia de aparición de cada valor, no su tamaño.

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Contextos apropiados para la moda</p>

La moda es la medida indicada cuando la pregunta es "¿cuál es el más común?":<br><br>
<b>Gestión de inventario en retail:</b> talla de ropa más vendida, color más popular, marca más comprada.<br><br>
<b>E-commerce:</b> categoría de producto más visitada, método de pago preferido, día con más compras.<br><br>
<b>Telecomunicaciones:</b> plan de datos más contratado, franja horaria de mayor actividad.<br><br>
<b>Salud:</b> tipo sanguíneo más frecuente (banco de sangre), diagnóstico más común en urgencias.<br><br>
<b>Desarrollo de software:</b> funcionalidad más utilizada (priorización), error más reportado (depuración), navegador más usado (compatibilidad).
</div>

## 3. Comparación y criterios de selección

### 3.1 Tabla comparativa

| Característica | Media | Mediana | Moda |
|---|---|---|---|
| **Tipo de variable** | Solo numéricas | Numéricas u ordinales | Cualquier tipo |
| **Fórmula** | $\frac{1}{n}\sum x_i$ | Valor central ordenado | Más frecuente |
| **Outliers** | Muy sensible | **Robusta** | **Robusta** |
| **Unicidad** | Siempre única | Siempre única | Puede ser múltiple o inexistente |
| **Uso de datos** | Todos los valores | Solo posición | Solo frecuencias |
| **Eficiencia** | Alta | Media | Baja |
| **Interpretación** | Centro de gravedad | Punto medio | Valor más común |
| **Complejidad** | $O(n)$ | $O(n \log n)$ | $O(n)$ |
| **Mejor uso** | Simétricas sin outliers | Asimétricas o con outliers | Variables nominales |

### 3.2 Diagrama de decisión

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")

def _b(x, y, w, h, t, fc, tc="white", fs=9):
    ax.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h,
                 boxstyle="round,pad=0.02,rounding_size=0.10",
                 fc=fc, ec="white", lw=1.5, zorder=2))
    ax.text(x, y, t, ha="center", va="center", color=tc,
            fontsize=fs, fontweight="bold", zorder=3)

def _arrow(x1, y1, x2, y2, lbl="", dx=0.0):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color="#555", lw=1.6), zorder=1)
    if lbl:
        ax.text((x1 + x2)/2 + dx, (y1 + y2)/2, lbl, fontsize=8, ha="center",
                color=UNAB_NAVY, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none"))

_b(5, 9, 2.6, 0.9, "Analizar dataset", "#2e6fb0")
_b(5, 6.9, 2.8, 0.9, "¿Tipo de variable?", "#d98a00")
_b(1.8, 4.6, 2.6, 1.0, "MODA\n(nominal)", "#2e8b57")
_b(6.6, 4.6, 3.0, 0.9, "¿Outliers o\nasimetría?", "#d98a00")
_b(4.8, 2.2, 2.4, 1.0, "MEDIANA\n(robusta)", "#2e8b57")
_b(8.4, 2.2, 2.4, 1.0, "MEDIA\n(eficiente)", "#2e8b57")

_arrow(5, 8.55, 5, 7.35)
_arrow(4.3, 6.6, 2.3, 5.1, "Nominal", dx=-0.3)
_arrow(5.7, 6.6, 6.4, 5.05, "Numérica", dx=0.4)
_arrow(6.2, 4.15, 5.1, 2.7, "Sí", dx=-0.2)
_arrow(7.2, 4.15, 8.2, 2.7, "No", dx=0.2)

ax.text(1.8, 3.7, "Preferencias, categorías", fontsize=7.5, ha="center", color="#555", style="italic")
ax.text(4.8, 1.4, "Salarios, precios, tiempos", fontsize=7.5, ha="center", color="#555", style="italic")
ax.text(8.4, 1.4, "Notas, datos balanceados", fontsize=7.5, ha="center", color="#555", style="italic")
ax.set_title("¿Qué medida usar?", color=UNAB_NAVY, fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

### 3.3 Relación entre las tres medidas como diagnóstico

La relación entre media, mediana y moda proporciona información directa sobre la **forma** de la distribución, sin necesidad de cálculos adicionales.

In [ ]:
# Generamos tres distribuciones y marcamos media, mediana y moda en cada una
rng = np.random.default_rng(3)
sim = rng.normal(50, 10, 50_000)                       # simétrica
pos = rng.lognormal(mean=3.2, sigma=0.5, size=50_000)  # asimetría positiva
neg = 100 - rng.lognormal(mean=3.2, sigma=0.5, size=50_000)  # asimetría negativa

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
casos = [(sim, "Simétrica",            r"$Me = \bar{x} = Mo$"),
         (pos, "Asimétrica positiva",  r"$Mo < Me < \bar{x}$"),
         (neg, "Asimétrica negativa",  r"$\bar{x} < Me < Mo$")]

for ax, (d, titulo, rel) in zip(axes, casos):
    ax.hist(d, bins=80, density=True, color="#bcd3ea", edgecolor="none")
    kde = stats.gaussian_kde(d)
    xs = np.linspace(d.min(), d.max(), 400)
    ax.plot(xs, kde(xs), color=UNAB_NAVY, lw=1.5)
    media, mediana = d.mean(), np.median(d)
    moda = xs[np.argmax(kde(xs))]              # moda ≈ pico de la densidad
    ax.axvline(moda,    color="#d98a00", ls=":",  lw=2, label="Moda")
    ax.axvline(mediana, color="#2e8b57", ls="--", lw=2, label="Mediana")
    ax.axvline(media,   color=UNAB_RED,  ls="-",  lw=2, label="Media")
    ax.set_title(f"{titulo}\n{rel}", fontsize=10, color=UNAB_NAVY)
    ax.set_yticks([]); ax.set_xticks([])

axes[0].legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

<div style="background-color:#fff4e6; border-left:5px solid #e65100; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#bf360c; font-size:1.05em;">⭐&nbsp; Importante — Interpretación de la relación entre las tres medidas</p>

<b>Si $\bar{x} \approx Me \approx Mo$:</b> distribución simétrica. La media es apropiada y eficiente. Los datos están bien balanceados.<br><br>
<b>Si $\bar{x} > Me > Mo$:</b> asimetría positiva (cola hacia la derecha). La media está elevada por valores extremos superiores. Reportar la mediana como indicador del valor típico.<br><br>
<b>Si $\bar{x} < Me < Mo$:</b> asimetría negativa (cola hacia la izquierda). La media está reducida por valores extremos inferiores. Reportar la mediana como indicador del valor típico.<br><br>
<b>Si $|\bar{x} - Me| > 0{,}20 \times Me$:</b> diferencia superior al 20 %. Indica presencia significativa de outliers o asimetría pronunciada. Investigar antes de reportar, considerar transformaciones (logaritmo, raíz cuadrada) y no usar la media sin una advertencia explícita.
</div>

| Relación | Tipo | Acción recomendada |
|---|---|---|
| $\bar{x} = Me = Mo$ | Simétrica | Usar media. Distribución equilibrada. |
| $Mo < Me < \bar{x}$ | Sesgo positivo | Cola derecha. Valores muy altos presentes. Usar mediana. |
| $\bar{x} < Me < Mo$ | Sesgo negativo | Cola izquierda. Valores muy bajos presentes. Usar mediana. |
| $\lvert\bar{x}-Me\rvert$ grande | Outliers | Investigar. La media no es confiable como único indicador. |

## 4. Implementación con NumPy

### 4.1 Cálculo básico y comparación entre medidas

In [ ]:
# Tiempos de entrega en días (incluye un caso excepcional)
tiempos = np.array([2, 3, 2, 4, 3, 2, 5, 3, 2, 45])

print("=" * 60)
print("ANÁLISIS DE TIEMPOS DE ENTREGA")
print("=" * 60)

media   = np.mean(tiempos)
mediana = np.median(tiempos)

print(f"\nMedia:   {media:.2f} días")
print(f"Mediana: {mediana:.2f} días")

# La moda requiere scipy (NumPy no la calcula)
moda_result = stats.mode(tiempos, keepdims=True)
moda = moda_result.mode[0]
freq = moda_result.count[0]
print(f"Moda:    {moda:.2f} días (aparece {freq} veces)")

# Diagnóstico automático
diff_pct = abs(media - mediana) / mediana * 100
print(f"\nDiferencia media-mediana: {diff_pct:.1f}%")

if diff_pct > 20:
    print("ALERTA: Diferencia >20% — presencia probable de outliers")
    print("Medida recomendada: MEDIANA")
else:
    print("Diferencia <20% — distribución aceptable")
    print("Medida recomendada: MEDIA")

<div style="background-color:#fffceb; border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#f57f17; font-size:1.05em;">📝&nbsp; Nota — Interpretación</p>

La media de 7,1 días está fuertemente inflada por el envío excepcional de 45 días. La mediana de 3 días representa con mayor fidelidad el tiempo que experimenta el cliente típico. La diferencia de 136,7 % entre ambas medidas es una señal clara de un outlier significativo. Reportar únicamente la media induciría a pensar que el servicio es más lento de lo que realmente es para la mayoría de los clientes.
</div>

### 4.2 Manejo de valores faltantes

En conjuntos de datos reales, la presencia de valores faltantes (`NaN`) es frecuente. NumPy provee versiones específicas de sus funciones estadísticas que los ignoran automáticamente.

In [ ]:
ventas = np.array([120, 135, np.nan, 142, 138, np.nan, 150, 145])

print(f"Datos originales: {ventas}")
print(f"Cantidad de NaN: {np.isnan(ventas).sum()}")

# Funciones nan-aware de NumPy
media_nan   = np.nanmean(ventas)
mediana_nan = np.nanmedian(ventas)
print(f"\nMedia (ignorando NaN):   {media_nan:.2f}")
print(f"Mediana (ignorando NaN): {mediana_nan:.2f}")

# Sin las versiones nan-aware, el resultado es NaN
print(f"\nMedia regular:   {np.mean(ventas)}")   # devuelve nan
print(f"Mediana regular: {np.median(ventas)}")   # devuelve nan

# Otros estadísticos nan-aware
print(f"\nDesv. estándar: {np.nanstd(ventas):.2f}")
print(f"Mínimo:         {np.nanmin(ventas):.2f}")
print(f"Máximo:         {np.nanmax(ventas):.2f}")

<div style="background-color:#fff4e6; border-left:5px solid #e65100; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#bf360c; font-size:1.05em;">⭐&nbsp; Importante — Funciones nan-aware de NumPy</p>

<ul>
<li><code>np.nanmean()</code>, <code>np.nanmedian()</code>: media y mediana ignorando NaN.</li>
<li><code>np.nanstd()</code>, <code>np.nanvar()</code>: desviación estándar y varianza.</li>
<li><code>np.nanmin()</code>, <code>np.nanmax()</code>: mínimo y máximo.</li>
<li><code>np.nanpercentile()</code>: percentiles.</li>
</ul>
<b>Buena práctica:</b> siempre reportar cuántos NaN existen y qué porcentaje representan. Si los valores faltantes superan el 10 % de los datos, las conclusiones estadísticas pueden ser cuestionables y deben acompañarse de una advertencia explícita.
</div>

### 4.3 Arrays multidimensionales

In [ ]:
# Ventas trimestrales por región (filas = regiones, columnas = trimestres)
ventas_matriz = np.array([
    [120, 135, 142, 138],   # Norte
    [98,  105, 112, 108],   # Sur
    [150, 148, 155, 160],   # Centro
    [85,  90,  88,  92]     # Oriente
])

print("Ventas por región y trimestre (miles USD):")
print(ventas_matriz)
print(f"Dimensiones: {ventas_matriz.shape}")

# Promedio anual por región (axis=1 -> por fila)
media_region = np.mean(ventas_matriz, axis=1)
regiones = ['Norte', 'Sur', 'Centro', 'Oriente']
print("\nPromedio anual por región:")
for region, val in zip(regiones, media_region):
    print(f"  {region:10s}: ${val:.2f}K")

# Promedio por trimestre (axis=0 -> por columna)
media_trim = np.mean(ventas_matriz, axis=0)
print("\nPromedio nacional por trimestre:")
for trim, val in zip(['Q1', 'Q2', 'Q3', 'Q4'], media_trim):
    print(f"  {trim}: ${val:.2f}K")

print(f"\nPromedio global: ${np.mean(ventas_matriz):.2f}K")

### 4.4 Percentiles y cuartiles

In [ ]:
edades = np.array([22, 25, 28, 30, 32, 35, 38, 40, 42, 45,
                   48, 50, 52, 55, 58, 60, 62, 65, 68, 70])

print(f"Análisis de percentiles (n = {len(edades)})")

Q1 = np.percentile(edades, 25)
Q2 = np.percentile(edades, 50)   # = mediana
Q3 = np.percentile(edades, 75)
IQR = Q3 - Q1

print(f"\nCuartiles:")
print(f"  Q1 (percentil 25): {Q1:.1f} años")
print(f"  Q2 (percentil 50 = Mediana): {Q2:.1f} años")
print(f"  Q3 (percentil 75): {Q3:.1f} años")
print(f"  IQR: {IQR:.1f} años")
print(f"  El 50% central tiene edades entre {Q1:.0f} y {Q3:.0f} años")

print(f"\nOtros percentiles:")
print(f"  P10: {np.percentile(edades, 10):.1f} — solo 10% es más joven")
print(f"  P90: {np.percentile(edades, 90):.1f} — solo 10% es mayor")
print(f"  P95: {np.percentile(edades, 95):.1f} — 5% más antiguo")

## 5. Implementación con pandas

pandas es la herramienta estándar cuando los datos contienen variables de tipos mixtos (numéricos, categóricos, fechas) organizadas en `DataFrames`.

### 5.1 Operaciones básicas

In [ ]:
datos = {
    'empleado_id':     [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'departamento':    ['Ventas','TI','Ventas','TI','RRHH',
                        'Ventas','TI','RRHH','Ventas','TI'],
    'calificacion':    [8.5, 9.2, 7.8, 9.5, 8.0, 8.5, 9.0, 8.5, 7.5, 9.3],
    'anos_experiencia':[3, 7, 2, 10, 5, 4, 8, 6, 1, 9],
    'salario':         [45, 75, 38, 95, 52, 48, 80, 58, 35, 88]
}

df = pd.DataFrame(datos)

# Estadísticos descriptivos
print(df[['calificacion', 'anos_experiencia', 'salario']].describe())

# Media, mediana y moda por columna
cols_num = ['calificacion', 'anos_experiencia', 'salario']
print("\nMedia por columna:")
print(df[cols_num].mean().round(2))

print("\nMediana por columna:")
print(df[cols_num].median().round(2))

print("\nModa de departamento:")
print(df['departamento'].mode()[0])

print("\nFrecuencia por departamento:")
print(df['departamento'].value_counts())

### 5.2 Análisis por grupos con `groupby`

<div style="background-color:#fffceb; border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#f57f17; font-size:1.05em;">📝&nbsp; Nota — Sobre el nombre de la variable</p>

En el apunte original esta tabla se guarda en una variable llamada <code>stats</code>. En un notebook de un solo kernel eso <b>sobrescribiría</b> el módulo <code>scipy.stats</code> que importamos al inicio (y que usa <code>stats.mode()</code>), dejando un error difícil de rastrear si más adelante vuelves a usar scipy. Por eso aquí la tabla se llama <code>tabla_stats</code>. La salida es idéntica.
</div>

In [ ]:
# Estadísticos de salario por departamento
print("Salario promedio por departamento:")
sal_medio = df.groupby('departamento')['salario'].mean().sort_values(ascending=False)
for dept, sal in sal_medio.items():
    print(f"  {dept:10s}: ${sal:,.2f}K")

# Múltiples estadísticos simultáneos (renombrado: tabla_stats, no 'stats')
print("\nEstadísticos completos de salario por departamento:")
tabla_stats = df.groupby('departamento')['salario'].agg([
    ('Media',   'mean'),
    ('Mediana', 'median'),
    ('Desv.Est','std'),
    ('Min',     'min'),
    ('Max',     'max'),
    ('N',       'count')
]).round(2)
print(tabla_stats)

# Múltiples columnas y funciones
stats_multi = df.groupby('departamento').agg({
    'calificacion':     ['mean', 'median'],
    'anos_experiencia': ['mean', 'median'],
    'salario':          ['mean', 'median', 'min', 'max']
}).round(2)
print("\nEstadísticos múltiples por departamento:")
print(stats_multi)

### 5.3 Manejo de valores faltantes en pandas

In [ ]:
df_na = df.copy()
df_na.loc[2, 'salario']          = np.nan
df_na.loc[5, 'anos_experiencia'] = np.nan
df_na.loc[7, 'calificacion']     = np.nan

print("Valores faltantes por columna:")
print(df_na.isnull().sum())

# pandas ignora NaN por defecto (skipna=True)
print(f"\nMedia salario (skipna=True):  ${df_na['salario'].mean():.2f}K")
print(f"Datos utilizados: {df_na['salario'].count()}/{len(df_na)}")

# Para incluir NaN explícitamente
print(f"Media salario (skipna=False): {df_na['salario'].mean(skipna=False)}")

# Análisis del impacto de los valores faltantes
print("\nImpacto de valores faltantes:")
for col in ['calificacion', 'anos_experiencia', 'salario']:
    media_orig = df[col].mean()
    media_na   = df_na[col].mean()
    diff_pct   = abs(media_orig - media_na) / media_orig * 100
    n_missing  = df_na[col].isnull().sum()
    print(f"  {col}: dif={diff_pct:.2f}%, faltantes={n_missing}/{len(df_na)}")

## 6. Ejemplo integrador: análisis de precios de viviendas

El siguiente ejemplo ilustra un flujo de trabajo completo, desde los estadísticos globales hasta la recomendación fundamentada sobre qué medida utilizar.

In [ ]:
np.random.seed(42)
n = 1000

df_viviendas = pd.DataFrame({
    'zona': np.random.choice(
        ['Centro', 'Norte', 'Sur', 'Periferia'],
        n, p=[0.2, 0.3, 0.3, 0.2]),
    'tipo': np.random.choice(
        ['Casa', 'Departamento', 'Duplex'],
        n, p=[0.4, 0.5, 0.1]),
    'metros_cuadrados': np.random.randint(50, 250, n),
    'dormitorios': np.random.choice([1, 2, 3, 4], n, p=[0.15, 0.35, 0.35, 0.15])
})

precio_base_m2 = {'Centro': 3000, 'Norte': 2200, 'Sur': 1800, 'Periferia': 1200}
df_viviendas['precio'] = df_viviendas.apply(
    lambda row: precio_base_m2[row['zona']] * row['metros_cuadrados']
                * (1 + np.random.normal(0, 0.15)), axis=1)

# Agregar propiedades de lujo
idx_lujo = np.random.choice(df_viviendas.index, size=20, replace=False)
df_viviendas.loc[idx_lujo, 'precio'] *= np.random.uniform(2, 4, size=20)

# 1. Estadísticos globales
media   = df_viviendas['precio'].mean()
mediana = df_viviendas['precio'].median()
diff_pct = abs(media - mediana) / mediana * 100

print("=" * 70)
print("ANÁLISIS DE PRECIOS DE VIVIENDAS")
print("=" * 70)
print(f"\nMedia:    ${media:,.0f}")
print(f"Mediana:  ${mediana:,.0f}")
print(f"Diferencia media-mediana: {diff_pct:.1f}%")

if diff_pct > 15:
    print("ALERTA: Diferencia >15% — asimetría o outliers significativos")

# 2. Por zona
print("\nEstadísticos por zona:")
stats_zona = df_viviendas.groupby('zona')['precio'].agg(
    Media='mean', Mediana='median', N='count').round(0)
print(stats_zona)

# 3. Detección de outliers (regla IQR de Tukey)
Q1 = df_viviendas['precio'].quantile(0.25)
Q3 = df_viviendas['precio'].quantile(0.75)
IQR = Q3 - Q1
lim_sup = Q3 + 1.5 * IQR
outliers = df_viviendas[df_viviendas['precio'] > lim_sup]
print(f"\nOutliers detectados: {len(outliers)} ({len(outliers)/n*100:.1f}%)")
print(f"Límite superior (Q3 + 1.5×IQR): ${lim_sup:,.0f}")

# 4. Comparación con y sin outliers
df_sin = df_viviendas[df_viviendas['precio'] <= lim_sup]
media_sin   = df_sin['precio'].mean()
mediana_sin = df_sin['precio'].median()
print(f"\nImpacto de outliers en la media:   {abs(media-media_sin)/media_sin*100:.1f}%")
print(f"Impacto de outliers en la mediana: {abs(mediana-mediana_sin)/mediana_sin*100:.1f}%")
print("La mediana es considerablemente más robusta.")

# 5. Recomendación
print(f"\nMedida representativa recomendada: MEDIANA = ${mediana:,.0f}")
print("Interpretación: el 50% de las viviendas tiene un precio inferior a este valor.")

*Visualización de soporte (histograma + boxplot), aplicando la buena práctica de "visualizar antes de decidir":*

In [ ]:
precio = df_viviendas['precio'].values

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 4.6),
                             height_ratios=[3, 1], sharex=True)
a1.hist(precio, bins=60, color="#bcd3ea", edgecolor="white")
a1.axvline(media,   color=UNAB_RED,    lw=2,          label=f"Media ${media:,.0f}")
a1.axvline(mediana, color="#2e8b57",   lw=2, ls="--", label=f"Mediana ${mediana:,.0f}")
a1.set_ylabel("Frecuencia"); a1.legend()
a1.set_title("Precios de viviendas: cola derecha por propiedades de lujo",
             color=UNAB_NAVY)

a2.boxplot(precio, widths=0.6, patch_artist=True,
           boxprops=dict(facecolor="#cbe6d4"), **HBOX_KW)
a2.set_yticks([]); a2.set_xlabel("Precio (USD)")
plt.tight_layout(); plt.show()

## 7. Aplicaciones por sector

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Finanzas: retornos de portafolio</p>

Un fondo de inversión analiza 120 meses (10 años) de retornos para comunicarlos a inversionistas potenciales. Los resultados son: media del 0,8 % mensual y mediana del 0,6 % mensual (diferencia del 33 %). La media está elevada por algunos meses excepcionales con retornos muy superiores al promedio habitual.<br><br>
<b>Decisión:</b> reportar ambas medidas con transparencia, utilizando la mediana para proyecciones conservadoras y la media para escenarios optimistas. Comunicar explícitamente la razón de la diferencia evita expectativas incorrectas de los inversionistas.
</div>

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — E-commerce: definición del umbral de envío gratuito</p>

Un análisis de 10.000 transacciones revela: media \$450 (inflada por pedidos corporativos), mediana \$85 (cliente típico), moda \$60 (carrito más frecuente), percentil 75 \$120. El 70 % de las transacciones es inferior a \$100.<br><br>
<b>Decisión:</b> fijar el umbral de envío gratuito en el rango \$85–\$95, que captura más del 50 % de las transacciones automáticamente y genera un incentivo realista para aumentar el valor del carrito. Establecer el umbral en la media (\$450) habría beneficiado únicamente al 15 % de los clientes.
</div>

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Recursos Humanos: análisis de equidad salarial</p>

Una empresa con 500 empleados (480 staff, 15 managers, 5 ejecutivos) reporta inicialmente un salario medio global de \$95K y concluye que la compensación es competitiva. Sin embargo, al calcular la mediana se obtienen \$68K, valor que se encuentra un 8 % por debajo del mercado.<br><br>
La causa: los salarios ejecutivos (\$300K–\$500K) elevan artificialmente la media global. El análisis segmentado por nivel revela que el staff gana en torno al mercado, pero la comparación global usando la media ocultaba la situación real. La corrección de esta discrepancia permitió resolver un problema de rotación de personal.
</div>

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Salud: tiempos de espera en urgencias</p>

Un análisis de 5.000 pacientes muestra: media 75 min, mediana 45 min, percentil 90 (P90) 120 min. El 50 % de los pacientes espera menos de 45 minutos, pero el 10 % espera más de 120 minutos y concentra la mayoría de las quejas.<br><br>
<b>Decisión:</b> utilizar la mediana (45 min) como comunicación estándar al público, el P90 (120 min) como indicador de compromiso de servicio formal, y no utilizar la media (75 min) que sobreestimaba el tiempo de espera típico. La mejora en la comunicación redujo las quejas en un 30 % sin modificar las operaciones.
</div>

## 8. Guía de selección

| Contexto | Medida | Justificación |
|---|---|---|
| Variable categórica nominal | **Moda** | Única medida aplicable. |
| Distribución simétrica sin outliers | **Media** | Eficiente, utiliza toda la información. |
| Distribución asimétrica con outliers | **Mediana** | Robusta; representa el valor típico. |
| Tiempos de respuesta / latencia | **Mediana + P90, P95** | Valor típico + compromisos de servicio. |
| Salarios e ingresos | **Mediana** | Los extremos altos distorsionan la media. |
| Precios de vivienda | **Mediana** | Las propiedades de lujo no dominan el indicador. |
| Proyecciones financieras | **Media + IC** | Eficiencia estadística con cuantificación de incertidumbre. |
| Control de calidad | **Media** | Procesos controlados con distribución estable. |
| Calificaciones académicas | **Media** | Distribución típicamente simétrica. |
| Preferencias de producto | **Moda** | La opción más popular es el dato relevante. |
| Edad de clientes homogéneos | **Media** | Distribución unimodal sin outliers. |
| Edad de clientes segmentados | **Mediana** | Posible bimodalidad; la media no es representativa. |

## 9. Buenas prácticas

<div style="background-color:#fff4e6; border-left:5px solid #e65100; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#bf360c; font-size:1.05em;">⭐&nbsp; Importante — Recomendaciones para el análisis de tendencia central</p>

<ol>
<li><b>Calcular siempre las tres medidas.</b> Una sola puede ser engañosa. La comparación entre media, mediana y moda revela la forma de la distribución sin cálculos adicionales.</li>
<li><b>Visualizar antes de decidir.</b> Un histograma y un boxplot toman pocos minutos y permiten identificar asimetría, outliers y multimodalidad antes de seleccionar la medida de reporte.</li>
<li><b>Contextualizar los resultados.</b> No basta con "el salario medio es \$X". La forma correcta incluye la medida elegida, su justificación, la dispersión (IQR o desviación estándar) y el tamaño de la muestra.</li>
<li><b>Reportar los valores faltantes.</b> Indicar cuántos NaN existen, qué porcentaje representan y qué tratamiento se les dio. Una tasa superior al 10 % puede comprometer las conclusiones.</li>
<li><b>Mantener la integridad estadística.</b> Si la media y la mediana difieren más del 15 %, explicar la razón. Si la medida elegida favorece un argumento, verificar que sea realmente la apropiada.</li>
<li><b>Utilizar percentiles estratégicamente.</b> P90, P95 y P99 son útiles para definir SLAs; los cuartiles (Q1, Q2, Q3) caracterizan la dispersión.</li>
<li><b>Adaptar la comunicación a la audiencia.</b> Con audiencias técnicas, especificar la medida e incluir la dispersión. Con audiencias no técnicas, reemplazar "mediana" por "valor típico" y usar analogías visuales.</li>
</ol>
</div>

### 9.1 Checklist de análisis

| ✓ | # | Acción |
|---|---|---|
| ☐ | 1 | Calcular media, mediana y moda |
| ☐ | 2 | Comparar media y mediana (¿diferencia $> 20\%$?) |
| ☐ | 3 | Construir histograma y boxplot |
| ☐ | 4 | Identificar outliers (regla IQR o z-scores) |
| ☐ | 5 | Calcular cuartiles y percentiles clave (P90, P95) |
| ☐ | 6 | Evaluar asimetría mediante la relación entre medidas |
| ☐ | 7 | Considerar el contexto del problema y la audiencia |
| ☐ | 8 | Seleccionar la medida con justificación explícita |
| ☐ | 9 | Calcular medidas de dispersión (desv. est., IQR) |
| ☐ | 10 | Analizar por grupos relevantes (`groupby`) |
| ☐ | 11 | Verificar valores faltantes y cuantificar su impacto |
| ☐ | 12 | Documentar todas las decisiones metodológicas |
| ☐ | 13 | Construir visualizaciones de soporte |
| ☐ | 14 | Redactar interpretación contextualizada |

*Función de apoyo que aplica el diagnóstico completo (pasos 1–6 del checklist) a cualquier serie numérica:*

In [ ]:
def diagnostico_tendencia_central(x, nombre="datos"):
    """Calcula las tres medidas, su relación y detecta outliers."""
    x = pd.Series(x).dropna()
    media   = x.mean()
    mediana = x.median()
    modas   = x.mode().tolist()
    Q1, Q3  = x.quantile(0.25), x.quantile(0.75)
    IQR     = Q3 - Q1
    out     = x[(x < Q1 - 1.5*IQR) | (x > Q3 + 1.5*IQR)]
    diff    = abs(media - mediana) / mediana * 100 if mediana != 0 else float('nan')

    print(f"--- Diagnóstico de '{nombre}' (n = {len(x)}) ---")
    print(f"Media:   {media:.2f}")
    print(f"Mediana: {mediana:.2f}")
    print(f"Moda(s): {modas}")
    print(f"P90: {x.quantile(0.90):.2f}   P95: {x.quantile(0.95):.2f}")
    print(f"Diferencia media-mediana: {diff:.1f}%")

    if diff < 5:
        forma = "≈ simétrica  ->  la MEDIA es apropiada"
    elif media > mediana:
        forma = "asimetría positiva (cola derecha)  ->  reportar MEDIANA"
    else:
        forma = "asimetría negativa (cola izquierda)  ->  reportar MEDIANA"
    print(f"Forma: {forma}")
    print(f"Outliers (1.5*IQR): {len(out)}")
    return None

# Demostración con los tiempos de entrega de la sección 4
diagnostico_tendencia_central([2, 3, 2, 4, 3, 2, 5, 3, 2, 45], "tiempos de entrega")

### Cierre

<div style="background-color:#ffffff; border-left:5px solid #0d2741; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d2741; font-size:1.05em;">💡&nbsp; Síntesis</p>

Las medidas de tendencia central son herramientas de cálculo simple, pero su valor radica en el <b>criterio</b> para seleccionar la apropiada y en la capacidad de interpretar lo que comunican. La media, la mediana y la moda no son intercambiables: cada una captura una dimensión distinta de los datos y es adecuada para contextos específicos. Proyectos que reportan el promedio cuando deberían reportar la mediana, o que ignoran la moda en variables categóricas, incurren en errores conceptuales con consecuencias prácticas. El dominio de estas medidas, combinado con el uso de dispersión, visualización y contexto, es el punto de partida de todo análisis estadístico riguroso.
</div>

---

## Referencias bibliográficas

- Freedman, D., Pisani, R., & Purves, R. (2007). *Statistics* (4th ed.). W. W. Norton.
- McKinney, W. (2022). *Python for data analysis* (3rd ed.). O'Reilly Media.
- Moore, D. S., McCabe, G. P., & Craig, B. A. (2017). *Introduction to the practice of statistics* (9th ed.). W. H. Freeman.
- Tukey, J. W. (1977). *Exploratory data analysis*. Addison-Wesley.
- VanderPlas, J. (2016). *Python data science handbook*. O'Reilly Media.

---
<div style="text-align:center; color:#0d2741;"><em>Estadística Computacional para la Toma de Decisiones · MCDIA · Universidad Andrés Bello · 2026</em></div>